In [2]:
from langchain_community.vectorstores import FAISS
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import TextLoader

In [9]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ-API-KEY"] = os.getenv("GROQ_API_KEY")


In [ ]:
loader = TextLoader("mmr_practice.txt")
docs = loader.load()

[Document(metadata={'source': 'mmr_practice.txt'}, page_content="Retrieval Augmented Generation, commonly called RAG, combines information retrieval with a Large Language Model. RAG retrieves relevant information from external documents and provides that information to the language model before generating an answer. A RAG system first searches for relevant documents and then uses the retrieved information to generate a response.\n\nRAG systems are useful because language models may not contain information about private or newly created documents. By retrieving information from an external knowledge base, RAG can answer questions using information that was not present in the original model training data.\n\nDocuments used in a RAG system are usually divided into smaller chunks. These chunks are converted into numerical representations called embeddings. Embeddings capture the semantic meaning of text and allow similar pieces of information to be found using vector similarity.\n\nA vecto

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 300, chunk_overlap = 50)
chunks = splitter.split_documents(docs)
chunks

In [7]:
embedddings = HuggingFaceEmbeddings(
    model_name = "all-MiniLM-L6-v2"
)
vectorestore = FAISS.from_documents(chunks,embedddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
retriever = vectorestore.as_retriever(
    search_type = "mmr",
    search_kwargs = {"k":3}
)


In [21]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
    You are a helpful assistant.

    Use the following context to answer the question.

    Context:
    {context}

    Question:
    {input}

    Answer:
    """
)

llm = init_chat_model(
    model="groq:openai/gpt-oss-120b"
)

In [22]:
documnet_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=prompt
)
rag_chain = create_retrieval_chain(
    retriever=retriever,
    combine_docs_chain=documnet_chain
)

In [26]:
query = {
    "input": "What is the difference between dense, sparse and hybrid retrieval?"
}

response = rag_chain.invoke(query)

print("\n" + "=" * 70)
print("                    RAG RESPONSE")
print("=" * 70)
print(response["answer"])






                    RAG RESPONSE
**Dense retrieval**  
- Represents queries and documents as continuous‑vector embeddings (often produced by neural encoders).  
- Retrieval is performed by measuring vector similarity (e.g., cosine or inner‑product).  
- Excels at capturing semantic similarity, so it can match a query with documents that use different wording but convey the same meaning.

**Sparse retrieval**  
- Treats text as a bag‑of‑words (or bag‑of‑ngrams) and scores documents with classic term‑weighting formulas such as BM25.  
- Relies heavily on exact keyword overlap and term frequencies.  
- Works best when the important information is expressed with specific terms, IDs, product names, or other precise tokens.

**Hybrid retrieval**  
- Combines the strengths of both approaches.  
- A typical hybrid system runs a dense retriever and a sparse retriever in parallel (or merges their index representations) and then fuses the results—e.g., by re‑ranking, weighted scoring, or union o